In [ ]:
!pip install ultralytics gradio requests -q

from ultralytics import YOLO

model = YOLO('/kaggle/input/datasets/amgdotexe/platecalc-weights/best.pt')
print("Model loaded successfully")
print("Sample classes:", [model.names[i] for i in range(5)])

In [ ]:
COUNTABLE_FOODS = {
    'egg', 'apple', 'banana', 'orange', 'pear', 'peach',
    'kiwi', 'mango', 'cherry', 'strawberry', 'grape',
    'dumpling', 'cookie', 'meatball'
}

UNIT_WEIGHTS = {
    'egg': 50, 'apple': 182, 'banana': 118, 'orange': 131,
    'pear': 166, 'peach': 150, 'kiwi': 69, 'mango': 200,
    'cherry': 8, 'strawberry': 12, 'grape': 5,
    'dumpling': 20, 'cookie': 14, 'meatball': 30
}

In [ ]:
import requests

def get_calories_per_100g(food_name: str) -> float:
    url = "https://api.nal.usda.gov/fdc/v1/foods/search"
    params = {
        "query": food_name,
        "pageSize": 1,
        "api_key": "DEMO_KEY"
    }
    try:
        resp = requests.get(url, params=params, timeout=5).json()
        foods = resp.get("foods", [])
        if not foods:
            return 0
        for nutrient in foods[0].get("foodNutrients", []):
            if nutrient.get("nutrientName") == "Energy":
                return nutrient.get("value", 0)
    except:
        return 0
    return 0

In [ ]:
import cv2
import numpy as np

def estimate_calories(image_path: str, model) -> dict:
    results = model(image_path)[0]
    total_calories = 0
    breakdown = []

    if results.masks is None:
        return {"total": 0, "items": [], "message": "No food detected"}

    img = cv2.imread(image_path)
    total_pixels = img.shape[0] * img.shape[1]

    for i, (mask, box) in enumerate(zip(results.masks.data, results.boxes)):
        class_id = int(box.cls)
        food_name = model.names[class_id]
        confidence = float(box.conf)

        if confidence < 0.3:
            continue

        mask_np = mask.cpu().numpy().astype(np.uint8)
        food_pixels = int(mask_np.sum())

        if food_name in COUNTABLE_FOODS:
            num_labels, _ = cv2.connectedComponents(mask_np)
            count = max(1, num_labels - 1)
            weight_g = count * UNIT_WEIGHTS.get(food_name, 100)
            method = f"{count} unit(s) × {UNIT_WEIGHTS.get(food_name, 100)}g"
        else:
            weight_g = (food_pixels / total_pixels) * 500
            method = f"{food_pixels} pixels → {weight_g:.0f}g"

        cal_per_100g = get_calories_per_100g(food_name)
        calories = (weight_g / 100) * cal_per_100g

        total_calories += calories
        breakdown.append({
            "food": food_name,
            "confidence": f"{confidence:.0%}",
            "weight_g": round(weight_g, 1),
            "cal_per_100g": cal_per_100g,
            "calories": round(calories, 1),
            "method": method
        })

    return {"total": round(total_calories, 1), "items": breakdown}

In [ ]:
import gradio as gr

def analyze_plate(image):
    if image is None:
        return "Please upload an image"
    
    result = estimate_calories(image, model)
    
    if not result["items"]:
        return "No food detected in this image"
    
    output = f"### Total Calories: {result['total']} kcal\n\n"
    output += "| Food | Confidence | Weight | Cal/100g | Calories |\n"
    output += "|------|------------|--------|----------|----------|\n"
    
    for item in result["items"]:
        output += f"| {item['food']} | {item['confidence']} | {item['weight_g']}g | {item['cal_per_100g']} | {item['calories']} kcal |\n"
    
    return output

demo = gr.Interface(
    fn=analyze_plate,
    inputs=gr.Image(type="filepath", label="Upload food image"),
    outputs=gr.Markdown(label="Calorie Breakdown"),
    title="PlateCalc",
    description="Upload a photo of your meal to estimate calories"
)

demo.launch()